In [ ]:
# Cell 1 — Evaluation Framework Setup & Rule-Based Evaluation
# Building systematic evaluation for prompt engineering

import boto3
import json
import time
from string import Template
from datetime import datetime

# ── Bedrock client ────────────────────────────────────────
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

def ask(prompt, system=None, max_tokens=1024, temperature=0.0):
    kwargs = {
        'modelId': model_id,
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system:
        kwargs['system'] = [{'text': system}]
    resp = bedrock.converse(**kwargs)
    return {
        'text':          resp['output']['message']['content'][0]['text'],
        'input_tokens':  resp['usage']['inputTokens'],
        'output_tokens': resp['usage']['outputTokens'],
        'stop_reason':   resp['stopReason']
    }

def parse_llm_json(text):
    """Strip markdown fencing and parse JSON from LLM output."""
    raw = text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)


# ── Test claims for evaluation ────────────────────────────
TEST_CLAIMS = [
    {
        "id": "EVAL-001",
        "description": """
        Policyholder: James Morton, Policy: AUTO-2024-88321
        Date of Loss: 2024-11-15
        Vehicle: 2022 Honda Accord, VIN: 1HGCV1F34NA012345
        
        Mr. Morton reports he was rear-ended at a red light on Highway 401 
        near Milton, Ontario. The other driver admitted fault at the scene. 
        Police report #2024-PLR-445566 filed. Damage to rear bumper, trunk, 
        and tail lights. Mr. Morton reports neck pain and visited the ER same day.
        Estimated vehicle damage: $8,500. Medical bills to date: $2,300.
        """,
        "expected_severity": "medium",
        "expected_liability": "third_party",
        "has_injury": True,
        "has_police_report": True
    },
    {
        "id": "EVAL-002",
        "description": """
        Policyholder: Sarah Kim, Policy: HOME-2024-55678
        Date of Loss: 2024-12-01
        Property: 45 Maple Drive, Oakville, ON
        
        Ms. Kim reports a kitchen fire caused by a faulty dishwasher. 
        Fire department responded and contained the fire to the kitchen.
        Fire report #FD-2024-8899 filed. Significant damage to kitchen 
        cabinets, countertops, appliances, and smoke damage throughout 
        first floor. No injuries. Temporary housing needed for estimated 
        3 weeks during repairs. Contractor estimate: $45,000.
        """,
        "expected_severity": "high",
        "expected_liability": "first_party",
        "has_injury": False,
        "has_police_report": False
    },
    {
        "id": "EVAL-003",
        "description": """
        Policyholder: David Chen, Policy: AUTO-2024-91234
        Date of Loss: 2024-12-10
        Vehicle: 2023 Tesla Model 3, VIN: 5YJ3E1EA8NF123456
        
        Mr. Chen reports his parked vehicle was vandalized overnight in a 
        public parking garage. Scratches along both sides, broken driver 
        side mirror, and slashed front tire. Security camera footage 
        requested but not yet obtained. Police report #2024-PLR-667788 filed.
        No injuries. Estimated damage: $3,200.
        """,
        "expected_severity": "low",
        "expected_liability": "comprehensive",
        "has_injury": False,
        "has_police_report": True
    }
]


# ── Prompt template for claim analysis ────────────────────
ANALYSIS_PROMPT = Template("""Analyze the following insurance claim and provide a structured assessment.

Claim Details:
$claim_description

Provide your analysis as JSON with exactly these fields:
{
    "claim_id": "the policy or claim number",
    "severity": "low | medium | high | critical",
    "liability_type": "first_party | third_party | comprehensive | uninsured_motorist",
    "injury_present": true/false,
    "police_report_filed": true/false,
    "estimated_total": numeric dollar amount,
    "key_findings": ["finding 1", "finding 2", "finding 3"],
    "recommended_action": "approve | investigate | deny | escalate",
    "confidence_score": 0.0 to 1.0,
    "reasoning": "brief explanation of your assessment"
}

Return ONLY the JSON object, no other text.""")


# ── Run analysis on all test claims ───────────────────────
print("=" * 70)
print("GENERATING CLAIM ANALYSES FOR EVALUATION")
print("=" * 70)

results = []
for claim in TEST_CLAIMS:
    prompt = ANALYSIS_PROMPT.substitute(claim_description=claim['description'])
    response = ask(prompt, max_tokens=1024, temperature=0.0)
    
    try:
        parsed = parse_llm_json(response['text'])
        results.append({
            'claim': claim,
            'response': response,
            'parsed': parsed,
            'parse_success': True
        })
        print(f"\n✓ {claim['id']}: Successfully parsed")
        print(f"  Severity: {parsed.get('severity', 'N/A')}")
        print(f"  Liability: {parsed.get('liability_type', 'N/A')}")
        print(f"  Action: {parsed.get('recommended_action', 'N/A')}")
        print(f"  Tokens: {response['input_tokens']} in / {response['output_tokens']} out")
    except json.JSONDecodeError as e:
        results.append({
            'claim': claim,
            'response': response,
            'parsed': None,
            'parse_success': False
        })
        print(f"\n✗ {claim['id']}: JSON parse failed — {e}")


# ── Rule-Based Evaluation ─────────────────────────────────
print("\n\n" + "=" * 70)
print("RULE-BASED EVALUATION")
print("=" * 70)

def evaluate_rules(result):
    """Apply automated rule checks to a parsed claim analysis."""
    scores = {}
    parsed = result['parsed']
    claim = result['claim']
    
    if not parsed:
        return {'parse_success': False, 'total_score': 0.0}
    
    # Rule 1: JSON parsed successfully
    scores['json_valid'] = True
    
    # Rule 2: All required fields present
    required_fields = [
        'claim_id', 'severity', 'liability_type', 'injury_present',
        'police_report_filed', 'estimated_total', 'key_findings',
        'recommended_action', 'confidence_score', 'reasoning'
    ]
    missing = [f for f in required_fields if f not in parsed]
    scores['fields_complete'] = len(missing) == 0
    scores['missing_fields'] = missing
    
    # Rule 3: Values in valid ranges
    valid_severity = parsed.get('severity') in ['low', 'medium', 'high', 'critical']
    valid_liability = parsed.get('liability_type') in [
        'first_party', 'third_party', 'comprehensive', 'uninsured_motorist'
    ]
    valid_action = parsed.get('recommended_action') in [
        'approve', 'investigate', 'deny', 'escalate'
    ]
    valid_confidence = isinstance(parsed.get('confidence_score'), (int, float)) and \
                       0 <= parsed.get('confidence_score', -1) <= 1
    scores['values_valid'] = all([valid_severity, valid_liability, valid_action, valid_confidence])
    
    # Rule 4: Factual accuracy checks against known data
    scores['severity_correct'] = parsed.get('severity') == claim['expected_severity']
    scores['liability_correct'] = parsed.get('liability_type') == claim['expected_liability']
    scores['injury_correct'] = parsed.get('injury_present') == claim['has_injury']
    scores['police_report_correct'] = parsed.get('police_report_filed') == claim['has_police_report']
    
    # Rule 5: Key findings is a non-empty list
    findings = parsed.get('key_findings', [])
    scores['has_findings'] = isinstance(findings, list) and len(findings) >= 2
    
    # Rule 6: Reasoning is substantive (not just a few words)
    reasoning = parsed.get('reasoning', '')
    scores['reasoning_substantive'] = len(reasoning) > 50
    
    # Calculate total score
    check_fields = [
        'json_valid', 'fields_complete', 'values_valid',
        'severity_correct', 'liability_correct', 'injury_correct',
        'police_report_correct', 'has_findings', 'reasoning_substantive'
    ]
    passed = sum(1 for f in check_fields if scores.get(f, False))
    scores['total_score'] = passed / len(check_fields)
    scores['passed'] = passed
    scores['total_checks'] = len(check_fields)
    
    return scores


# Run rule-based evaluation
all_scores = []
for result in results:
    scores = evaluate_rules(result)
    all_scores.append(scores)
    claim_id = result['claim']['id']
    
    print(f"\n{'─' * 50}")
    print(f"  {claim_id}: {scores.get('passed', 0)}/{scores.get('total_checks', 0)} checks passed "
          f"({scores.get('total_score', 0):.0%})")
    print(f"{'─' * 50}")
    
    if scores.get('total_score', 0) == 0:
        print(f"  ✗ Parse failed — no evaluation possible")
        continue
        
    for key, value in scores.items():
        if key in ('total_score', 'passed', 'total_checks', 'missing_fields'):
            continue
        status = "✓" if value else "✗"
        print(f"  {status} {key}: {value}")
    
    if scores.get('missing_fields'):
        print(f"  ⚠ Missing fields: {scores['missing_fields']}")

# Summary
avg_score = sum(s.get('total_score', 0) for s in all_scores) / len(all_scores)
print(f"\n{'=' * 70}")
print(f"RULE-BASED EVALUATION SUMMARY")
print(f"{'=' * 70}")
print(f"  Average score: {avg_score:.0%}")
print(f"  Claims evaluated: {len(all_scores)}")
print(f"  Perfect scores: {sum(1 for s in all_scores if s.get('total_score') == 1.0)}")
print(f"  Parse failures: {sum(1 for s in all_scores if not s.get('json_valid', False))}")

In [ ]:
# Cell 2 — LLM-as-Judge Evaluation
# Using a second LLM call to evaluate response quality

JUDGE_PROMPT = Template("""You are an experienced insurance claims quality reviewer. 
Your job is to evaluate the quality of an AI-generated claim analysis.

ORIGINAL CLAIM:
$claim_description

AI-GENERATED ANALYSIS:
$analysis_json

Evaluate the analysis on these five dimensions. Score each 1-5 
(1=poor, 3=acceptable, 5=excellent):

1. COMPLETENESS: Does the analysis address all relevant aspects of the claim?
   - Are injuries, property damage, and liability properly identified?
   - Are important details from the claim captured in the findings?

2. ACCURACY: Are the factual determinations correct?
   - Is the severity level appropriate for the claim value and circumstances?
   - Is the liability classification correct?
   - Are boolean fields (injury, police report) factually accurate?

3. REASONING: Is the recommended action well-supported?
   - Does the reasoning logically connect evidence to conclusion?
   - Would an experienced adjuster agree with this recommendation?

4. ACTIONABILITY: Could a claims handler act on this analysis?
   - Are the key findings specific enough to guide next steps?
   - Is the recommended action clear and appropriate?

5. RISK AWARENESS: Does the analysis identify potential issues?
   - Are subrogation opportunities noted where relevant?
   - Are fraud indicators or red flags mentioned if applicable?
   - Are coverage concerns or exclusions flagged?

Provide your evaluation as JSON:
{
    "completeness": {"score": 1-5, "explanation": "brief explanation"},
    "accuracy": {"score": 1-5, "explanation": "brief explanation"},
    "reasoning": {"score": 1-5, "explanation": "brief explanation"},
    "actionability": {"score": 1-5, "explanation": "brief explanation"},
    "risk_awareness": {"score": 1-5, "explanation": "brief explanation"},
    "overall_score": weighted average (accuracy=30%, reasoning=25%, completeness=20%, actionability=15%, risk_awareness=10%),
    "critical_issues": ["list any significant problems"],
    "improvement_suggestions": ["list specific improvements"]
}

Return ONLY the JSON object.""")


# ── Run LLM-as-Judge on each result ──────────────────────
print("=" * 70)
print("LLM-AS-JUDGE EVALUATION")
print("=" * 70)

judge_results = []
for result in results:
    if not result['parse_success']:
        print(f"\n✗ {result['claim']['id']}: Skipped — parse failure")
        judge_results.append(None)
        continue
    
    # Build the judge prompt
    prompt = JUDGE_PROMPT.substitute(
        claim_description=result['claim']['description'],
        analysis_json=json.dumps(result['parsed'], indent=2)
    )
    
    # Use a system prompt to reinforce the judge role
    judge_system = """You are a senior quality assurance reviewer for an insurance 
    claims processing system. You evaluate AI-generated analyses with the same 
    rigor you would apply to a junior adjuster's work. Be fair but thorough — 
    identify both strengths and weaknesses. Score conservatively: a 3 means 
    'acceptable for production use' and a 5 means 'exceptional, no improvements needed.'"""
    
    judge_response = ask(prompt, system=judge_system, max_tokens=1024, temperature=0.0)
    
    try:
        judge_parsed = parse_llm_json(judge_response['text'])
        judge_results.append({
            'scores': judge_parsed,
            'tokens': {
                'input': judge_response['input_tokens'],
                'output': judge_response['output_tokens']
            }
        })
        
        print(f"\n{'─' * 50}")
        print(f"  {result['claim']['id']}: JUDGE SCORES")
        print(f"{'─' * 50}")
        
        for dim in ['completeness', 'accuracy', 'reasoning', 'actionability', 'risk_awareness']:
            score_data = judge_parsed.get(dim, {})
            score = score_data.get('score', 'N/A')
            bar = "█" * int(score) + "░" * (5 - int(score)) if isinstance(score, (int, float)) else "N/A"
            print(f"  {dim:<18} {bar}  {score}/5")
            print(f"    → {score_data.get('explanation', 'No explanation')}")
        
        overall = judge_parsed.get('overall_score', 'N/A')
        print(f"\n  {'Overall':<18} {overall}/5.0")
        
        # Show critical issues if any
        issues = judge_parsed.get('critical_issues', [])
        if issues and issues != ['None'] and issues != ['none']:
            print(f"\n  ⚠ Critical Issues:")
            for issue in issues:
                print(f"    • {issue}")
        
        # Show improvement suggestions
        suggestions = judge_parsed.get('improvement_suggestions', [])
        if suggestions:
            print(f"\n  💡 Suggestions:")
            for s in suggestions:
                print(f"    • {s}")
                
    except json.JSONDecodeError as e:
        print(f"\n✗ {result['claim']['id']}: Judge response parse failed — {e}")
        judge_results.append(None)


# ── Judge Evaluation Summary ──────────────────────────────
print(f"\n\n{'=' * 70}")
print("LLM-AS-JUDGE SUMMARY")
print("=" * 70)

valid_judges = [j for j in judge_results if j is not None]
if valid_judges:
    # Average scores per dimension
    dimensions = ['completeness', 'accuracy', 'reasoning', 'actionability', 'risk_awareness']
    print(f"\n  Average Scores Across All Claims:")
    for dim in dimensions:
        scores = [j['scores'].get(dim, {}).get('score', 0) for j in valid_judges]
        avg = sum(scores) / len(scores) if scores else 0
        bar = "█" * int(round(avg)) + "░" * (5 - int(round(avg)))
        print(f"    {dim:<18} {bar}  {avg:.1f}/5")
    
    # Overall averages
    overalls = [j['scores'].get('overall_score', 0) for j in valid_judges]
    avg_overall = sum(overalls) / len(overalls) if overalls else 0
    print(f"\n    {'Overall Average':<18}        {avg_overall:.2f}/5.0")
    
    # Total judge tokens
    total_judge_in = sum(j['tokens']['input'] for j in valid_judges)
    total_judge_out = sum(j['tokens']['output'] for j in valid_judges)
    print(f"\n  Judge Token Cost:")
    print(f"    Input:  {total_judge_in:,} tokens")
    print(f"    Output: {total_judge_out:,} tokens")
    print(f"    Total:  {total_judge_in + total_judge_out:,} tokens")

In [ ]:
# Cell 3 — Human Evaluation Framework & Combined Scoring
# Structured rubrics for human reviewers + unified scoring across all three methods

# ── Human Evaluation Rubric ───────────────────────────────
print("=" * 70)
print("HUMAN EVALUATION FRAMEWORK")
print("=" * 70)

HUMAN_RUBRIC = {
    "professional_quality": {
        "description": "Would you send this analysis to a senior adjuster as-is?",
        "scale": {
            1: "Needs complete rewrite — inaccurate or unprofessional",
            2: "Major issues — several corrections needed before use",
            3: "Acceptable — minor edits needed but core analysis is sound",
            4: "Good — ready for use with minimal tweaks",
            5: "Excellent — production-ready, no changes needed"
        }
    },
    "domain_accuracy": {
        "description": "Are insurance-specific determinations correct?",
        "scale": {
            1: "Fundamental misunderstanding of coverage or liability",
            2: "Multiple incorrect determinations",
            3: "Core determinations correct, minor issues",
            4: "All determinations correct with good detail",
            5: "Expert-level analysis with nuanced understanding"
        }
    },
    "completeness": {
        "description": "Does the analysis cover everything a handler needs?",
        "scale": {
            1: "Missing critical information",
            2: "Several important gaps",
            3: "Covers the basics adequately",
            4: "Thorough with good coverage of key factors",
            5: "Comprehensive — nothing an adjuster would add"
        }
    },
    "actionability": {
        "description": "Can a claims handler take immediate next steps from this?",
        "scale": {
            1: "No clear path forward",
            2: "Vague recommendations",
            3: "General direction is clear",
            4: "Specific and actionable next steps",
            5: "Detailed action plan with priorities"
        }
    }
}

# Display the rubric
for dimension, details in HUMAN_RUBRIC.items():
    print(f"\n  {dimension.upper()}")
    print(f"  {details['description']}")
    for score, desc in details['scale'].items():
        print(f"    {score}: {desc}")


# ── Simulated Human Scores ────────────────────────────────
# In production, these would come from a review UI or spreadsheet
# For demonstration, we provide realistic scores a reviewer might give

print(f"\n\n{'=' * 70}")
print("SIMULATED HUMAN REVIEW SCORES")
print("=" * 70)

HUMAN_SCORES = {
    "EVAL-001": {
        "professional_quality": 4,
        "domain_accuracy": 4,
        "completeness": 4,
        "actionability": 3,
        "reviewer_notes": "Good analysis of rear-end collision. Correctly identified "
                         "third-party liability. Could improve by specifying next steps "
                         "for BI claim handling and subrogation timeline."
    },
    "EVAL-002": {
        "professional_quality": 4,
        "domain_accuracy": 3,
        "completeness": 4,
        "actionability": 4,
        "reviewer_notes": "Strong analysis overall. Incorrectly flagged fire report as "
                         "police report — this matters for documentation requirements. "
                         "Good identification of temporary housing needs and scope of damage."
    },
    "EVAL-003": {
        "professional_quality": 4,
        "domain_accuracy": 5,
        "completeness": 3,
        "actionability": 3,
        "reviewer_notes": "Accurate severity and liability classification. Could be more "
                         "detailed on next steps — should mention requesting security footage, "
                         "getting photos of damage, and checking for witnesses."
    }
}

for claim_id, scores in HUMAN_SCORES.items():
    avg = sum(v for k, v in scores.items() if isinstance(v, int)) / 4
    print(f"\n{'─' * 50}")
    print(f"  {claim_id}: Human Average {avg:.1f}/5.0")
    print(f"{'─' * 50}")
    for dim, score in scores.items():
        if dim == 'reviewer_notes':
            print(f"\n  Notes: {score}")
        else:
            bar = "█" * score + "░" * (5 - score)
            print(f"  {dim:<22} {bar}  {score}/5")


# ── Combined Evaluation Dashboard ─────────────────────────
print(f"\n\n{'=' * 70}")
print("COMBINED EVALUATION DASHBOARD")
print("=" * 70)
print(f"{'Method':<20} {'Weight':<10} {'Score':<10} {'Weighted':<10}")
print(f"{'─' * 50}")

# Rule-based score (from Cell 1)
rule_avg = sum(s.get('total_score', 0) for s in all_scores) / len(all_scores)

# LLM judge score (from Cell 2) — normalize to 0-1
valid_judges = [j for j in judge_results if j is not None]
judge_avg = sum(j['scores'].get('overall_score', 0) for j in valid_judges) / len(valid_judges) / 5.0

# Human score — normalize to 0-1
human_avgs = []
for scores in HUMAN_SCORES.values():
    avg = sum(v for k, v in scores.items() if isinstance(v, int)) / 4
    human_avgs.append(avg)
human_avg = sum(human_avgs) / len(human_avgs) / 5.0

# Weighted combination
weights = {'rule_based': 0.30, 'llm_judge': 0.30, 'human_review': 0.40}

rule_weighted = rule_avg * weights['rule_based']
judge_weighted = judge_avg * weights['llm_judge']
human_weighted = human_avg * weights['human_review']

print(f"{'Rule-Based':<20} {'30%':<10} {rule_avg:.2f}{'':>5} {rule_weighted:.3f}")
print(f"{'LLM-as-Judge':<20} {'30%':<10} {judge_avg:.2f}{'':>5} {judge_weighted:.3f}")
print(f"{'Human Review':<20} {'40%':<10} {human_avg:.2f}{'':>5} {human_weighted:.3f}")
print(f"{'─' * 50}")
combined = rule_weighted + judge_weighted + human_weighted
print(f"{'COMBINED SCORE':<20} {'100%':<10} {'':>10} {combined:.3f}")

# Quality gate
print(f"\n  Quality Gate: {'✓ PASS' if combined >= 0.75 else '✗ FAIL'} (threshold: 0.75)")

# Per-claim breakdown
print(f"\n\n{'=' * 70}")
print("PER-CLAIM BREAKDOWN")
print("=" * 70)
print(f"{'Claim':<12} {'Rule':<10} {'Judge':<10} {'Human':<10} {'Combined':<10} {'Status':<8}")
print(f"{'─' * 60}")

for i, result in enumerate(results):
    claim_id = result['claim']['id']
    
    r_score = all_scores[i].get('total_score', 0)
    
    j_score = 0
    if judge_results[i]:
        j_score = judge_results[i]['scores'].get('overall_score', 0) / 5.0
    
    h_scores = HUMAN_SCORES.get(claim_id, {})
    h_score = sum(v for k, v in h_scores.items() if isinstance(v, int)) / 4 / 5.0 if h_scores else 0
    
    combined_claim = (r_score * weights['rule_based'] + 
                      j_score * weights['llm_judge'] + 
                      h_score * weights['human_review'])
    
    status = "✓ PASS" if combined_claim >= 0.75 else "✗ FAIL"
    
    print(f"{claim_id:<12} {r_score:<10.2f} {j_score:<10.2f} {h_score:<10.2f} {combined_claim:<10.3f} {status}")


# ── Token Cost Summary ────────────────────────────────────
print(f"\n\n{'=' * 70}")
print("TOTAL TOKEN COST")
print("=' * 70}")

total_analysis_in = sum(r['response']['input_tokens'] for r in results)
total_analysis_out = sum(r['response']['output_tokens'] for r in results)
total_judge_in = sum(j['tokens']['input'] for j in valid_judges)
total_judge_out = sum(j['tokens']['output'] for j in valid_judges)

print(f"\n  {'Stage':<20} {'Input':<12} {'Output':<12} {'Total':<12}")
print(f"  {'─' * 56}")
print(f"  {'Analysis':<20} {total_analysis_in:<12,} {total_analysis_out:<12,} {total_analysis_in + total_analysis_out:<12,}")
print(f"  {'Judge':<20} {total_judge_in:<12,} {total_judge_out:<12,} {total_judge_in + total_judge_out:<12,}")
grand_in = total_analysis_in + total_judge_in
grand_out = total_analysis_out + total_judge_out
print(f"  {'─' * 56}")
print(f"  {'TOTAL':<20} {grand_in:<12,} {grand_out:<12,} {grand_in + grand_out:<12,}")

# Cost estimate (Claude Sonnet pricing)
input_cost = grand_in * 3.0 / 1_000_000
output_cost = grand_out * 15.0 / 1_000_000
print(f"\n  Estimated cost: ${input_cost + output_cost:.4f} per evaluation run")
print(f"  At 1000 claims/day: ${(input_cost + output_cost) * 1000:.2f}/day")

In [ ]:
# Cell 4 — Summary & Production Evaluation Strategy
# Tying together all three evaluation methods with production guidance

print("=" * 70)
print("DAY 26 SUMMARY: EVALUATION FRAMEWORK FOR PROMPT ENGINEERING")
print("=" * 70)

print("""
THREE-TIER EVALUATION PYRAMID
─────────────────────────────

                    ┌─────────┐
                    │  HUMAN  │  1% of claims
                    │ REVIEW  │  Gold standard calibration
                   ─┤         ├─
                  / └─────────┘ \\
                 /   Subjective   \\
                /    quality &     \\
               /     real-world     \\
              /      judgment        \\
             ┌───────────────────────┐
             │    LLM-AS-JUDGE      │  10% of claims
             │  Quality & reasoning  │  Scalable quality signal
            ─┤                      ├─
           / └───────────────────────┘ \\
          /   Semantic evaluation:      \\
         /    relevance, reasoning,      \\
        /     tone, completeness          \\
       ┌───────────────────────────────────┐
       │        RULE-BASED CHECKS         │  100% of claims
       │   Structure, schema, ranges       │  Instant, near-zero cost
       └───────────────────────────────────┘


KEY CONCEPTS
────────────

1. RULE-BASED EVALUATION (Cell 1)
   • Runs on every response — fast and deterministic
   • Checks: JSON validity, required fields, value ranges, 
     factual accuracy against known data
   • Think of it as: automated unit tests for LLM output
   • Catches: structural failures, missing data, invalid values
   • Cannot catch: bad reasoning, poor tone, missing context

2. LLM-AS-JUDGE (Cell 2)
   • Runs on a sample — costs tokens but scales
   • Evaluates: completeness, accuracy, reasoning, 
     actionability, risk awareness
   • Think of it as: automated peer review
   • Catches: weak reasoning, gaps in analysis, poor quality
   • Limitation: self-evaluation bias (same model family)
   • Mitigation: use different model, calibrate with examples

3. HUMAN REVIEW (Cell 3)
   • Runs on smallest sample — expensive but essential
   • Evaluates: professional quality, domain accuracy, 
     real-world usability
   • Think of it as: the gold standard calibration layer
   • Catches: everything the other two miss
   • Critical for: model updates, prompt changes, new claim types

4. COMBINED SCORING
   • Weighted: Rule-based 30% + LLM-judge 30% + Human 40%
   • Quality gate threshold determines pass/fail
   • Per-claim breakdown identifies specific problem areas
   • Convergence across methods = high confidence
   • Divergence across methods = investigation needed


PRODUCTION STRATEGY
───────────────────

  ┌──────────────┬────────────┬─────────────┬──────────────┐
  │ Tier         │ Coverage   │ Cost/Claim  │ Latency      │
  ├──────────────┼────────────┼─────────────┼──────────────┤
  │ Rule-based   │ 100%       │ ~$0.00      │ Milliseconds │
  │ LLM-judge    │ 10%        │ ~$0.02      │ 2-4 seconds  │
  │ Human review │ 1%         │ ~$5-15      │ Hours/Days   │
  └──────────────┴────────────┴─────────────┴──────────────┘

  When to increase sampling rates:
  • After model version updates
  • After prompt template changes  
  • After system prompt modifications
  • New claim types or lines of business
  • Quality scores trending downward


CONNECTING TO CI/CD (Day 24)
────────────────────────────

  The evaluation framework feeds directly into the prompt 
  regression testing pipeline:

  Prompt Change → Deploy to Staging
       ↓
  Run evaluation suite (rule-based on all test cases)
       ↓
  LLM-judge on full test suite (affordable for test sets)
       ↓
  Compare scores to baseline
       ↓
  Pass → Deploy to Production
  Fail → Block deployment, alert prompt engineer


TOKEN COST AWARENESS
────────────────────
""")

# Recalculate costs for display
analysis_tokens = sum(r['response']['input_tokens'] + r['response']['output_tokens'] for r in results)
judge_tokens = sum(j['tokens']['input'] + j['tokens']['output'] for j in valid_judges)

print(f"  This evaluation run:")
print(f"    Analysis generation: {analysis_tokens:,} tokens")
print(f"    Judge evaluation:    {judge_tokens:,} tokens")
print(f"    Total:               {analysis_tokens + judge_tokens:,} tokens")
print(f"    Evaluation overhead:  {judge_tokens/analysis_tokens:.1f}x the analysis cost")
print(f"""
  Key insight: Evaluation costs MORE than the analysis itself.
  This is expected and acceptable — you're investing in quality
  assurance. But it reinforces why you sample rather than 
  evaluate every response.


WHAT'S NEXT
───────────
  Day 27: Red Teaming & Adversarial Testing
  • Systematic attacks on your prompts
  • Jailbreak attempts and boundary violations  
  • Building adversarial test suites
  • Defense strategies and hardening
""")